# 02 — Solver debugging: reading the solved 5×3

Phase 3 leaves two complete databases on disk, 4.1 GB each, one per H2 arm.
This notebook opens them and answers the questions that came up repeatedly
during the phase and were each time answered by a throwaway script.

**Read this first.** A throwaway probe written during this phase accumulated
the search frontier in a `set` of `GameState`. `history` is part of that
dataclass's `__eq__`, so identical positions reached by different move orders
never deduplicated and the probe grew with the number of *paths*. It reached
12.5 GiB before it was killed. **Key positions by `(colours, hands, to_move)`
or by layer index — never by `GameState` identity.**

Requires the checkpoints. Run under PyPy for anything that walks a whole
layer; CPython is fine for single-position work.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from fliphex.moves import apply_move, legal_moves  # noqa: E402
from fliphex.notation import encode_move  # noqa: E402
from fliphex.rules import is_terminal, winner  # noqa: E402
from fliphex.variant import Arm, Variant  # noqa: E402
from solver.checkpoint import Checkpoint  # noqa: E402
from solver.minimax import LOSS, WIN  # noqa: E402
from solver.retrograde import LayerIndex  # noqa: E402
from solver.sweep_reader import SweepReader  # noqa: E402

ARM = "h1"
variant = Variant(5, 3, Arm(ARM))
board = variant.board()
reader = SweepReader(variant, Checkpoint(ROOT / "data/checkpoints" / variant.name))
hands = [len(h) for h in variant.deck_names()]
print(variant.name, variant.n_cells, "cells, hands", hands)

## 1. The root, and every first move

The value of the opening position, then the value of the position after each
distinct first move. This is exactly what the PV audit's part 1 re-derives by
forward search; here it is a table lookup, so the two are independent only
in the direction the audit runs.

In [ ]:
root = variant.initial_state()
print("root:", "P1 wins" if reader.value(root) == WIN else "P2 wins")

# Distinct *positions* after one ply, not distinct moves: several (tile,
# rotation) pairs collapse onto the same child (adr-003, and the 4.46x
# aliasing in adr-005's Phase 3 amendment).
children = {}
for move in legal_moves(board, root):
    child = apply_move(board, root, move)
    children.setdefault((child.colours, child.hands, child.to_move), move)

print(f"{len(legal_moves(board, root))} moves -> {len(children)} distinct positions")

In [ ]:
# Which first moves keep the win? A move is good iff it leaves the opponent lost.
good, bad = [], []
for _key, move in children.items():
    child = apply_move(board, root, move)
    (good if reader.value(child) == LOSS else bad).append(move)

print(f"winning first moves: {len(good)} of {len(children)} distinct positions")
for move in good[:10]:
    print(f"  {encode_move(board, move)}")

## 2. Walk the principal variation

One optimal line from the root to a terminal position. Every position on it
must alternate WIN / LOSS — if it does not, the database is inconsistent and
that is a finding, not a display bug.

In [ ]:
pv = reader.principal_variation(board, root)
state = root
for ply, move in enumerate(pv, start=1):
    value = "WIN " if reader.value(state) == WIN else "LOSS"
    print(f"  ply {ply:2d}  {value} to move  ->  {encode_move(board, move)}")
    state = apply_move(board, state, move)
print("  terminal:", winner(state).name, "| board full:", is_terminal(state))

## 3. The parity split, straight off the layers

Four instruments have now shown the same thing: sweep runtime, criticality,
block-RLE ratios, and the WIN/LOSS mix itself. This cell measures the mix
directly — a popcount per layer — which is the measurement the registry says
deserves an experiment of its own rather than more speculation.

**Run this under PyPy.** It touches all 17.5 × 10⁹ entries.

In [ ]:
from solver.retrograde import SLOT_WIN  # noqa: E402

MAX_LAYER = 6  # raise deliberately; t=9 alone is 5.02e9 entries

print(f"{'t':>3} {'size':>15} {'WIN %':>8}  mover")
for t in range(MAX_LAYER + 1):
    values = reader.layer(t)
    size = LayerIndex(variant, t).size
    wins = sum(
        1 for i in range(size) if ((values[i >> 2] >> ((i & 3) << 1)) & 3) == SLOT_WIN
    )
    mover = "P1" if t % 2 == 0 else "P2"
    print(f"{t:>3} {size:>15,} {100 * wins / size:7.2f}%  {mover}")
    reader.release()

## 4. Inspect one position by hand

The workhorse: given a position, show its value, every child's value, and
which moves preserve the win. This is what to reach for when an agent plays
something that looks wrong.

In [ ]:
def explain(state, limit=12):
    """Value of `state` and of every distinct child."""
    print(
        "to move:",
        state.to_move.name,
        "| value:",
        "WIN" if reader.value(state) == WIN else "LOSS",
    )
    seen = {}
    for move in legal_moves(board, state):
        child = apply_move(board, state, move)
        seen.setdefault((child.colours, child.hands, child.to_move), (move, child))
    wins = [(m, c) for m, c in seen.values() if reader.value(c) == LOSS]
    print(f"distinct children: {len(seen)}   winning for the mover: {len(wins)}")
    for move, _child in wins[:limit]:
        print(f"  {encode_move(board, move)}")


# Three plies down the PV, as an example.
probe = root
for move in pv[:3]:
    probe = apply_move(board, probe, move)
explain(probe)

## 5. Compare the two arms on one position

The criticality measure position by position. **The index correspondence is a
permutation, not identity** — `LayerIndex` orders a hand by tile index, so
P1's extra tile sits at position 7 in h1 and 6 in h2 and the shared `P6` moves
with it. `scripts/exp002_criticality.py` derives the map; do not compare raw
indices here.

In [ ]:
art = json.loads((ROOT / "results/exp002-criticality-5x3.json").read_text())
print(
    f"criticality: {100 * art['criticality']:.3f}%"
    f"  ({art['criticality_numerator']:,} of {art['criticality_denominator']:,})"
)
print(
    f"cross-arm verification: {art['verification_compared']:,} compared, "
    f"{art['verification_mismatches']:,} mismatches"
)
print()
print(f"{'t':>3} {'in hand':>15} {'critical':>13} {'crit %':>8}")
for row in art["layers"]:
    rate = 100 * row["critical"] / row["in_hand"] if row["in_hand"] else 0
    print(
        f"{row['layer']:>3} {row['in_hand']:>15,} {row['critical']:>13,} {rate:>7.2f}%"
    )